# Identity-bound benchmark manifests

This notebook walks a tracked v0.15 smoke fixture through
`koopman-graph benchmark run` and `verify`. You should leave knowing
how the canonical summary digest is formed (the full identity body,
not a short field subset), how `run` rejects a wrong dataset payload,
and how `verify` rejects an inconsistent `summary.json`.

**Honesty.** This path does **not** fit `GraphKoopmanModel`, does
**not** download METR-LA or PEMS HDF5, and does **not** invent forecast
MAE or RMSE. `run` writes `executed=False`. YAML manifests need
`pip install "koopman-graph[cli]"` (PyYAML).


## Setup

Locate the checkout, require PyYAML for the tracked YAML manifests, and
define a subprocess helper for `python -m koopman_graph.cli` (equivalent
to the `koopman-graph` console script). The notebook was written against
`koopman_graph` 0.15.0. Stored cell outputs are teaching evidence for
that version; a later bump changes the live `summary_sha256` because
`package_version` participates in the digest.


In [1]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import yaml  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "YAML manifests require PyYAML. Install with: "
        "pip install 'koopman-graph[cli]'"
    ) from exc

import koopman_graph


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        manifest = candidate / "benchmarks" / "v0.15" / "smoke_telemetry.yaml"
        if (candidate / "pyproject.toml").is_file() and manifest.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the KoopmanGraph checkout "
        "(need benchmarks/v0.15/smoke_telemetry.yaml)."
    )


ROOT = repo_root()
SMOKE = ROOT / "benchmarks" / "v0.15"
WORKDIR = Path(tempfile.mkdtemp(prefix="kg-bench-47-"))


def _public_text(text: str) -> str:
    """Replace checkout, workdir, and home prefixes in printed text."""
    for old, new in (
        (str(WORKDIR), "<workdir>"),
        (str(ROOT), "<repo>"),
        (str(Path.home()), "<home>"),
    ):
        text = text.replace(old, new)
    return text


def kg_cli(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    completed = subprocess.run(
        [sys.executable, "-m", "koopman_graph.cli", *args],
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.stdout:
        text = _public_text(completed.stdout)
        print(text, end="" if text.endswith("\n") else "\n")
    if completed.stderr:
        text = _public_text(completed.stderr)
        print(text, end="" if text.endswith("\n") else "\n")
    if check and completed.returncode != 0:
        completed.check_returncode()
    return completed


print(f"koopman_graph {koopman_graph.__version__}")
print("located checkout; workdir is temporary")


koopman_graph 0.15.0
located checkout; workdir is temporary


## Motivation and background

Ad-hoc training scripts do not pin dataset bytes, seeds, or method
roles. A frozen `ExperimentManifest` (`benchmark_manifest_v1`) records
that protocol. `run` checks the payload SHA-256 against
`dataset.sha256` (raw file bytes, not a hash of the YAML document).
It then writes a canonical SHA-256 over the summary body: UTF-8 JSON,
`sort_keys=True`, compact separators, `ensure_ascii=False`, with
`summary_sha256` omitted. The body fields are `schema_version`,
`manifest_id`, `manifest_sha256`, `dataset_sha256`, `track`, `methods`,
`seeds`, `horizons`, `metrics`, `controls`, `package_version`, and
`executed`. `verify` recomputes that digest and binds the summary to
the loaded manifest (id, track, dataset digest, method names and roles,
seeds, horizons, metrics, controls, and `manifest_sha256`). It does
not require `executed` to be `False`, and it does not require
`package_version` to equal the installed `koopman_graph` version.

The payloads under `benchmarks/v0.15/data/` are tiny UTF-8 stand-ins,
not the METR-LA HDF5 of Li et al. (2018) or Caltrans PeMS caches.
Example 22 remains the METR-LA teaching-baseline comparison; this
notebook does not re-run it and does not download METR-LA.


## Minimal example

`run` the telemetry smoke manifest (`smoke_telemetry.yaml`) against its
hashed payload into a temporary directory, then `verify` the written
`summary.json`. Expect exit code 0 and `executed=False`.


In [2]:
TELEMETRY_MANIFEST = SMOKE / "smoke_telemetry.yaml"
TELEMETRY_DATA = SMOKE / "data" / "smoke_telemetry.txt"
TELEMETRY_OUT = WORKDIR / "telemetry-run"

kg_cli(
    "benchmark",
    "run",
    "--manifest",
    str(TELEMETRY_MANIFEST),
    "--data",
    str(TELEMETRY_DATA),
    "--out",
    str(TELEMETRY_OUT),
)
kg_cli(
    "benchmark",
    "verify",
    "--manifest",
    str(TELEMETRY_MANIFEST),
    "--against",
    str(TELEMETRY_OUT),
)

live = json.loads((TELEMETRY_OUT / "summary.json").read_text(encoding="utf-8"))
assert live["executed"] is False
assert live["schema_version"] == "benchmark_summary_v1"
assert live["manifest_id"] == "smoke-telemetry"
print("executed=", live["executed"])
print("summary_sha256=", live["summary_sha256"])
print("metrics (declared names, not scores)=", live["metrics"])


wrote summary: <workdir>/telemetry-run/summary.json
verified summary: <workdir>/telemetry-run/summary.json
executed= False
summary_sha256= f912c7d86b7630e0d508f72de4e95dbdc515dc6b6bf79687d2b65fea56e8ec59
metrics (declared names, not scores)= ['mae', 'rmse']


## Progressive deep dive

`verify` the other two checked-in summaries (multiphysics, topology
transfer) without a second `run`. Then point `run` at a wrong payload
so the dataset SHA-256 check fails. Next, flip `executed` on a copy of
the telemetry summary while leaving `summary_sha256` stale, then
rewrite that digest so `verify` succeeds with `executed=True`. Finally,
repeat the telemetry `run` through `koopman_graph.benchmark.run_manifest`
(deep import, not the root façade).


In [3]:
checked_in = []
for track in ("smoke_multiphysics", "smoke_topology_transfer"):
    kg_cli(
        "benchmark",
        "verify",
        "--manifest",
        str(SMOKE / f"{track}.yaml"),
        "--against",
        str(SMOKE / "summaries" / f"{track}.json"),
    )
    payload = json.loads(
        (SMOKE / "summaries" / f"{track}.json").read_text(encoding="utf-8")
    )
    assert payload["executed"] is False
    checked_in.append((track, payload["summary_sha256"]))

for track, digest in checked_in:
    print(f"{track}: verified summary_sha256={digest}")


verified summary: <repo>/benchmarks/v0.15/summaries/smoke_multiphysics.json
verified summary: <repo>/benchmarks/v0.15/summaries/smoke_topology_transfer.json
smoke_multiphysics: verified summary_sha256=01d05d9d192df7af10cb2a0f1cbb341055d2cdb9fb523df5c81c18745b7da344
smoke_topology_transfer: verified summary_sha256=27ebe7ba9454e52770524b06af513bbfed2330e989c12cb856341150fda861e8


Point `run` at a temporary file whose bytes are not the hashed
telemetry payload. Expect a non-zero exit and a SHA-256 mismatch;
`run` must not write a summary for that data.


In [4]:
wrong_payload = WORKDIR / "wrong_telemetry.txt"
wrong_payload.write_text("not the hashed telemetry bytes\n", encoding="utf-8")
wrong_out = WORKDIR / "wrong-payload-run"
wrong = kg_cli(
    "benchmark",
    "run",
    "--manifest",
    str(TELEMETRY_MANIFEST),
    "--data",
    str(wrong_payload),
    "--out",
    str(wrong_out),
    check=False,
)
assert wrong.returncode != 0, "wrong payload must fail run"
assert not (wrong_out / "summary.json").is_file()
print("wrong-payload exit code=", wrong.returncode)


error: SHA256 mismatch for <workdir>/wrong_telemetry.txt: got de1f5986a8f80e6164fb0aaf605c997a3742b87affcedcaa7d51842c34617f46, expected bcb2f305aaa35fbb8a5f0b2caff14192a2d66613c5970e214b0127175b1b6aba
wrong-payload exit code= 1


Copy the checked-in telemetry summary and set `executed=True` without
recomputing `summary_sha256`. `verify` should fail with a digest
mismatch. That failure is an inconsistent hash, not a policy that
`executed` must stay `False`.


In [5]:
checked = json.loads(
    (SMOKE / "summaries" / "smoke_telemetry.json").read_text(encoding="utf-8")
)
tampered_path = WORKDIR / "tampered_telemetry.json"
tampered = dict(checked)
tampered["executed"] = True
tampered_path.write_text(json.dumps(tampered, indent=2) + "\n", encoding="utf-8")

failed = kg_cli(
    "benchmark",
    "verify",
    "--manifest",
    str(TELEMETRY_MANIFEST),
    "--against",
    str(tampered_path),
    check=False,
)
assert failed.returncode != 0, "stale summary_sha256 must fail verify"
print("stale-digest verify exit code=", failed.returncode)


error: summary_sha256 mismatch: got 0b6c25dfe9139c84fcf69f3032bc969f47ea55a7dd02b5249480864c3101d31d, expected 5bffaffd130eb793539fbf71efcbcedd354514f42c4502fb3a9a62f006e5d82e
stale-digest verify exit code= 1


Recompute `summary_sha256` over the canonical body with the public
`canonical_sha256` and `summary_to_mapping` helpers, then `verify`
again. A consistent `executed=True` document binds to the same
manifest and passes. `run` still writes `executed=False`; `verify`
does not enforce that flag.


In [6]:
from koopman_graph.benchmark import (
    canonical_sha256,
    summary_from_mapping,
    summary_to_mapping,
    verify_summary,
)

record = summary_from_mapping(tampered)
body = summary_to_mapping(record)
body.pop("summary_sha256")
rewritten = summary_to_mapping(record)
rewritten["summary_sha256"] = canonical_sha256(body)
rewritten_path = WORKDIR / "rewritten_executed_true.json"
rewritten_path.write_text(json.dumps(rewritten, indent=2) + "\n", encoding="utf-8")
rewritten_ok = verify_summary(TELEMETRY_MANIFEST, rewritten_path)
assert rewritten_ok.executed is True
print("rewritten executed=True verify ok")
print("rewritten summary_sha256=", rewritten_ok.summary_sha256)


rewritten executed=True verify ok
rewritten summary_sha256= 5bffaffd130eb793539fbf71efcbcedd354514f42c4502fb3a9a62f006e5d82e


The same telemetry `run` / `verify` path is available as a deep import
(`koopman_graph.benchmark.run_manifest`), not through the root package
façade.


In [7]:
from koopman_graph.benchmark import load_summary, run_manifest, verify_summary

api_out = WORKDIR / "telemetry-python"
written = run_manifest(TELEMETRY_MANIFEST, TELEMETRY_DATA, api_out)
summary = verify_summary(TELEMETRY_MANIFEST, written)
loaded = load_summary(written)
assert summary.executed is False
assert loaded.executed is False
print("python run_manifest wrote <workdir>/telemetry-python/summary.json")
print("executed=", summary.executed)
print("summary_sha256=", summary.summary_sha256)


python run_manifest wrote <workdir>/telemetry-python/summary.json
executed= False
summary_sha256= f912c7d86b7630e0d508f72de4e95dbdc515dc6b6bf79687d2b65fea56e8ec59


## Results

Identity fields from the live telemetry `run`. Look at `executed=False`
and at `metrics` as **declared names** (MAE, RMSE) rather than reported
scores. A passing `verify` on the checked-in JSON files is a hash check,
not a forecast table. The wrong-payload `run` and the stale digest fail
on purpose; the rewritten `executed=True` file is the control that
`verify` checks consistency, not the `executed` flag.


In [8]:
rows = [
    ("smoke_telemetry (live run)", live),
    (
        "smoke_telemetry (checked-in)",
        json.loads(
            (SMOKE / "summaries" / "smoke_telemetry.json").read_text(
                encoding="utf-8"
            )
        ),
    ),
]
print(f"{'source':<32} {'executed':<10} {'summary_sha256'}")
for name, payload in rows:
    print(f"{name:<32} {str(payload['executed']):<10} {payload['summary_sha256']}")
print("controls=", live["controls"])
print("methods=", live["methods"])
print("wrong-payload run exit=", wrong.returncode)
print("stale-digest verify exit=", failed.returncode)
print("rewritten executed=", rewritten_ok.executed)
print(
    "Live package_version may differ from a checked-in summary; "
    "verify does not require those versions to match."
)
shutil.rmtree(WORKDIR, ignore_errors=True)
print("cleaned workdir")


source                           executed   summary_sha256
smoke_telemetry (live run)       False      f912c7d86b7630e0d508f72de4e95dbdc515dc6b6bf79687d2b65fea56e8ec59
smoke_telemetry (checked-in)     False      0b6c25dfe9139c84fcf69f3032bc969f47ea55a7dd02b5249480864c3101d31d
controls= ['pernode']
methods= [{'name': 'graph_koopman', 'role': 'koopman'}, {'name': 'pernode_ls', 'role': 'control'}]
wrong-payload run exit= 1
stale-digest verify exit= 1
rewritten executed= True
Live package_version may differ from a checked-in summary; verify does not require those versions to match.
cleaned workdir


## Interpretation / discussion

A passing `verify` means the stored `summary_sha256` matches the
canonical body and the identity fields still bind to the loaded
manifest (including `dataset_sha256` and `manifest_sha256`). It is
not evidence of forecast skill and is not a LibCity (Wang et al.,
2021) or BasicTS (Shao et al., 2025) leaderboard result. Live `run`
records the installed `koopman_graph` version in `package_version`;
that field participates in the digest, so a version bump changes
`summary_sha256` without invalidating an older checked-in file.
`verify` does not require those versions to match and does not
require `executed=False`. Method execution (actual MAE / RMSE)
remains a later increment.

Use example 22 for the METR-LA teaching-baseline comparison of Li et
al. (2018) (unequal budgets, not dedicated-library SOTA). Use
examples 37 and 38 for transfer and factorization ablations,
including negative outcomes.


## Takeaways

- Reach for `koopman-graph benchmark run` / `verify` when you need a
  frozen, tamper-evident protocol hash.
- YAML manifests need `pip install "koopman-graph[cli]"`; JSON does not.
- `run` writes `executed=False` (no training and no invented scores).
  `verify` rejects an inconsistent digest or unbound identity fields;
  it does not enforce `executed=False`.
- `run` rejects a `--data` payload whose SHA-256 does not match
  `dataset.sha256`.
- Do not download METR-LA for this notebook; use `benchmarks/v0.15/`.
- Do not read declared `metrics` names as reported MAE / RMSE.
- The live digest includes `package_version`; stored outputs here are
  teaching evidence for 0.15.0.


## Further reading

- Sphinx: [Identity-bound benchmarks](https://koopmangraph.readthedocs.io/en/latest/benchmarks.html),
  [Command-line interface](https://koopmangraph.readthedocs.io/en/latest/cli.html),
  [API reference](https://koopmangraph.readthedocs.io/en/latest/api.html)
  (`ExperimentManifest`, `run_manifest`, `verify_summary`; deep import,
  not the root façade)
- Tracked fixtures: [`benchmarks/v0.15/README.md`](../benchmarks/v0.15/README.md)
- Wang et al. (2021), LibCity
  ([doi:10.1145/3474717.3483923](https://doi.org/10.1145/3474717.3483923))
- Shao et al. (2025), BasicTS
  ([doi:10.1109/TKDE.2024.3484454](https://doi.org/10.1109/TKDE.2024.3484454))
- Li et al. (2018), DCRNN / METR-LA
  ([OpenReview](https://openreview.net/forum?id=SJiHXGWAZ))
- Related notebooks:
  [`22_gnn_forecaster_comparison.ipynb`](22_gnn_forecaster_comparison.ipynb)
  (METR-LA teaching baselines),
  [`37_cross_topology_transfer.ipynb`](37_cross_topology_transfer.ipynb),
  [`38_operator_factorization_ablation.ipynb`](38_operator_factorization_ablation.ipynb)
